# What each prescribed BVOC species does in NorESM (OsloAero + tropstrat chemistry)
Companion to `plot_bvoc_emissions.ipynb`: the 19 MEGAN-fed species of the fBVOC runs, their emissions (Tg of species per year, from that notebook's section 2), and the processes they enter **in this model version**.

Sources read (NorESM2.3_beta01, `components/cam/src/chemistry/`):
- gas phase: `pp_tropstrat_mam_oslo/chem_mech.in` (+ `chem_mech.doc`, the compiled list); `mozart/` is the generic MOZART driver (no species-specific reactions; those are all in `chem_mech.in`);
- aerosol: `oslo_aero/src/oslo_aero_condtend.F90` (condensation, nucleation, apparent formation rate);
- run setup: `atm_in` of `NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-20261029` (`rad_climate`, `flbc_list`; identical to `-20261023`).

**Evidence tags:** **(a)** read in the code or namelist · **(b)** inference from the code + standard chemistry/literature (not diagnosed from our runs).

## 1. Model facts the table depends on (a)
| # | Fact | Where |
|---|---|---|
| F1 | **Only MTERP and ISOP make SOA** (plus DMS). The VBS yields for BCARY, TOLUENE, BENZENE, ISOP, MTERP are commented out (`*[..._vbs]`). | `chem_mech.in` l. 1130–1140 (commented), 1171–1178 (active) |
| F2 | MTERP + O₃ → **0.15 SOA_LV** (low volatility); MTERP + OH / NO₃ → **0.15 SOA_SV** (semi-volatile); ISOP + O₃ / OH / NO₃ → **0.025 SOA_SV** (molar yields). These are *tracer* reactions: they do not consume MTERP, ISOP or the oxidant; the rate is k·[VOC]·[oxidant]. | `chem_mech.in` l. 1173–1178 |
| F3 | SOA_LV and SOA_SV condense on all modes in proportion to the condensation sink (≈ particle surface; sticking 0.3–1). Condensed mass goes to the **SOA_A1** coating tracer. | `oslo_aero_condtend.F90` l. 123–143, 477–480 |
| F4 | **Nucleation**: only SOA_LV can nucleate, and only 50 % of what is left after condensation (`lvocfrac = 0.5`). SOA_SV never nucleates. Nucleated organic mass → **SOA_NA** (Aitken SO4/SOA mode). | l. 308, 399–401, 473–480 |
| F5 | **Boundary-layer nucleation** (Paasonen 2010, `pbl_nucleation = 2`): J = 6.1·10⁻⁷ [H₂SO₄] + 3.9·10⁻⁸ [ORG] (cm⁻³ s⁻¹), only below max(PBLH, 500 m) (≤ 7 km). Above: binary H₂SO₄–H₂O only (Vehkamäki 2002), no organic nucleation term. | l. 671–682, 881–892 |
| F6 | **Survival to the formation size** (Lehtinen 2007): J_x = J₁·exp(−γ·d₁·CoagS·3600 / GR), with GR = GR(H₂SO₄) + GR(ORG). Organics raise GR, so survival rises **exponentially**; the organic share of nucleated mass = GR_ORG / GR. GR is applied at **all levels**, so SOA_LV also helps binary-nucleated particles survive above the PBL. Caps: J ≤ 10¹⁰, J_x ≤ 10³ cm⁻³ s⁻¹. | l. 909–948, 966–1003 |
| F7 | Organic nitrates from terpenes and isoprene (TERPNIT, NTERPOOH, ISOPNITA/B, ONITR, …) hydrolyse on aerosol to HNO₃: a NOₓ sink that scales with aerosol surface. | `chem_mech.in` l. 1112–1125 |
| F8 | Radiation sees the model's **O₃ and CH₄** (`rad_climate = 'A:O3', 'A:CH4', …`). But **CH₄ and CO₂ have prescribed lower boundaries** (`flbc_list`), so the classic "VOC/CO eat OH → longer CH₄ lifetime → warming" feedback is **largely suppressed** in these runs (surface CH₄ is reset; only small changes aloft). CO → CO₂ is not radiatively active. | run `atm_in` |
| F9 | SOA_LV / SOA_SV are not in `gas_wetdep_list`: their only fate is condensation/nucleation. | run `atm_in` |

## 2. Table: emissions and processes per species
Columns:
- **Tg/yr**: global (CTRL_PD, CTRL_FUT) and LCC-modified cells (CTRL and LCC, PD and FUT), new-chain 2001–2014 climatologies (`bvoc_totals_tg_per_yr.csv` written by `plot_bvoc_emissions.ipynb`).
- **pathway**: SOA (aerosol) and/or gas-phase chemistry.
- **processes**: the specific route in the code.
- **direct / indirect**: *direct* = the species' own oxidation products enter the process; *indirect* = it acts only by changing something else (oxidants, NOₓ, PAN, aerosol surface).
- **linearity**: response of the process to a **small increase** of this species' emission, and when it stops being linear.
- **sign**: expected radiative effect of an **increase** of this species alone in this model, all else fixed: ❄ cooling, 🔥 warming, ≈ small/uncertain. All signs are **(b)**; the processes are **(a)** unless tagged.

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

TOTALS = Path("bvoc_totals_tg_per_yr.csv")   # written by plot_bvoc_emissions.ipynb, section 2
if not TOTALS.exists():
    raise FileNotFoundError(f"{TOTALS} missing: run plot_bvoc_emissions.ipynb first")
totals = pd.read_csv(TOTALS, index_col="species")

In [ ]:
# Shared text for the many small oxygenated / light VOCs (all (b) for the sign)
O3_NL = ("O₃ production ∝ VOC only where NOₓ is high (VOC-limited, polluted); in low-NOₓ boreal air "
         "it is NOₓ-limited, so extra VOC changes O₃ little (sublinear, saturating)")
ROWS = [
 dict(species="MTERP", name="monoterpenes (α-pinene, β-pinene, limonene, … lumped)", group="terpene",
      pathway="SOA + chemistry",
      processes="SOA: +O₃ → SOA_LV → (i) new-particle formation in the PBL (50 % of left-over LV) and (ii) coating; "
                "+OH/NO₃ → SOA_SV → coating only (SOA_A1), growing Aitken particles to CCN size. "
                "Chemistry: O₃ and OH sink (ozonolysis also gives OH); TERPO2/NTERPO2 → organic nitrates (NOₓ sink on aerosol, F7); "
                "products CH2O, acetone, TERPROD1/2.",
      direct_indirect="Direct (SOA from its own oxidation). Indirect: lowers OH and O₃ → less H₂SO₄ from SO₂+OH and less SOA from other precursors; removes NOₓ.",
      linearity="SOA production linear in [MTERP]·[oxidant] (F2) but oxidants are titrated when MTERP rises → sublinear at high emission. "
                "Nucleation J linear in ORG (F5), but survival exp(−γ·CoagS/GR) rises steeply with GR: strongest gain where GR is small and "
                "CoagS large (clean→moderately polluted PBL); self-limiting because extra SOA raises CoagS/condensation sink. "
                "The ORG nucleation term acts only below max(PBLH, 500 m); the GR (survival) boost acts at all levels. CCN → cloud response saturates (Twomey ∝ log N): largest in pristine boreal air.",
      sign="❄ cooling (SOA direct + aerosol–cloud); small 🔥 from O₃ where NOₓ is high",
      ref="chem_mech.in l. 1068–1070, 1173–1175; condtend l. 401, 473–480, 881–892, 1003"),
 dict(species="ISOP", name="isoprene", group="hemiterpene",
      pathway="SOA (small) + chemistry",
      processes="SOA: +O₃/OH/NO₃ → 0.025 SOA_SV → coating only (no nucleation). "
                "Chemistry: dominant OH sink over forests; ISOPAO2/ISOPBO2 + NO → NO₂ → O₃; MVK, MACR, CH2O, IEPOX, HPALD, PAN/MPAN (NOₓ export); "
                "isoprene nitrates → HNO₃ on aerosol; ozonolysis.",
      direct_indirect="Direct for SOA_SV coating (small yield). Mostly indirect: O₃ (production or loss), OH suppression "
                      "(less SO₂ → H₂SO₄ → fewer nucleated particles; slower MTERP+OH → less SOA_SV), PAN.",
      linearity="SOA_SV linear in [ISOP]·[ox] until OH is depleted (sublinear). "
                "O₃: " + O3_NL + "; at very low NOₓ, more ISOP lowers O₃ (ozonolysis, HO₂/RO₂ instead of NO). "
                "OH suppression is non-linear (OH recycling via HPALD only partly represented).",
      sign="≈ / 🔥 (O₃ where NOₓ high; less H₂SO₄ nucleation); small ❄ from SOA_SV. Weak CH₄ effect (F8)",
      ref="chem_mech.in l. 998–1013, 1176–1178"),
 dict(species="BCARY", name="β-caryophyllene (sesquiterpenes)", group="sesquiterpene",
      pathway="chemistry only (no SOA in this version!)",
      processes="Very fast ozonolysis (k = 1.2·10⁻¹⁴, lifetime ~minutes) → OH, HO2, TERPROD1/2, acetone, CH2O; +OH → TERPO2; +NO₃ → NTERPO2 (organic nitrates). "
                "VBS SOA yields exist in the file but are commented out (F1).",
      direct_indirect="Indirect only: local O₃ sink and OH source in the canopy/PBL. More OH → faster SO₂ → H₂SO₄ and MTERP+OH → SOA_SV; "
                      "less O₃ → slower MTERP+O₃ → less SOA_LV (the nucleating vapour).",
      linearity="Linear in [BCARY]·[O₃] while O₃ is not depleted; near sources O₃ consumption makes it sublinear. "
                "Opposite-sign effects on SOA_LV (−) and SOA_SV / H₂SO₄ (+) → net depends on the MTERP/BCARY ratio and O₃ level.",
      sign="≈ (small); in reality a strong SOA precursor (cooling), but not in this NorESM version",
      ref="chem_mech.in l. 1065–1067, 1130–1131 (commented)"),
 dict(species="C3H6", name="propene", group="alkene",
      pathway="chemistry", processes="+OH → PO2 → CH3CHO, CH2O (+ NO → NO₂ → O₃); +O₃ → OH, HCOOH, CH3COOH, CH3CHO; +NO₃ → NOA (organic nitrate).",
      direct_indirect="Indirect: O₃ production, OH sink/source, PAN via CH3CHO.", linearity=O3_NL + ".",
      sign="🔥 small (O₃)", ref="chem_mech.in l. 921–939"),
 dict(species="BIGENE", name="lumped C≥4 alkenes (butenes, …)", group="alkene",
      pathway="chemistry", processes="+OH → ENEO2 → CH3CHO, CH2O, acetone (+ NO → NO₂ → O₃); +NO₃ → NO₂ + carbonyls.",
      direct_indirect="Indirect: O₃ production, PAN via CH3CHO.", linearity=O3_NL + ".",
      sign="🔥 small (O₃)", ref="chem_mech.in l. 944–946"),
 dict(species="C2H4", name="ethene (ethylene)", group="alkene",
      pathway="chemistry", processes="+OH → EO2 → CH2O, glycolaldehyde (+ NO → NO₂ → O₃); +O₃ → CO, HCOOH, CH2O, OH.",
      direct_indirect="Indirect: O₃ production, OH sink.", linearity=O3_NL + ".",
      sign="🔥 small (O₃)", ref="chem_mech.in l. 889–890, 909–915"),
 dict(species="CH3CHO", name="acetaldehyde", group="aldehyde",
      pathway="chemistry", processes="+OH/NO₃ → CH3CO3 → **PAN** (+NO₂); photolysis → CH3O2 + CO + HO2. PAN carries NOₓ to remote/upper troposphere and releases it (thermal decomposition) → O₃ there.",
      direct_indirect="Indirect: PAN (NOₓ transport) → remote O₃; HOₓ source.",
      linearity="PAN formation needs NO₂ (∝ CH3CO3·NO₂, competes with CH3CO3+NO/HO₂) → saturates when NOₓ-limited; remote O₃ gain ∝ NOₓ exported, not ∝ CH3CHO.",
      sign="🔥 small (O₃, remote)", ref="chem_mech.in l. 606, 899–900, 916–917"),
 dict(species="CH2O", name="formaldehyde", group="aldehyde",
      pathway="chemistry", processes="Photolysis → H + HCO → **2 HO₂** (radical source) or H₂ + CO; +OH → CO + HO₂; +NO₃ → HNO₃. HO₂ + NO → NO₂ → O₃.",
      direct_indirect="Indirect: HOₓ source → O₃; slightly raises OH.",
      linearity="HOₓ source linear in CH2O; O₃ gain needs NO (NOₓ-limited → small). Secondary CH2O from ISOP/MTERP oxidation is far larger than this primary flux.",
      sign="🔥 small (O₃)", ref="chem_mech.in l. 604–605, 862–865"),
 dict(species="CH3OH", name="methanol", group="alcohol",
      pathway="chemistry", processes="+OH → CH2O + HO₂ (lifetime ~1–2 weeks); wet deposition.",
      direct_indirect="Indirect: OH sink, CH2O/HO₂ → O₃.", linearity="Linear (slow, well-mixed); " + O3_NL + ".",
      sign="🔥 very small (O₃); large flux but low reactivity", ref="chem_mech.in l. 870"),
 dict(species="C2H5OH", name="ethanol", group="alcohol",
      pathway="chemistry", processes="+OH → CH3CHO + HO₂ → PAN; wet deposition.",
      direct_indirect="Indirect: via CH3CHO → PAN → O₃.", linearity="Linear in OH sink; PAN/O₃ part NOₓ-limited (see CH3CHO).",
      sign="🔥 very small", ref="chem_mech.in l. 895"),
 dict(species="CH3COOH", name="acetic acid", group="organic acid",
      pathway="chemistry (minor)", processes="+OH → CH3O2 + CO₂ (slow, k = 7·10⁻¹³); mostly wet/dry deposition. No aerosol uptake in OsloAero.",
      direct_indirect="Indirect, negligible.", linearity="Linear.",
      sign="≈ 0", ref="chem_mech.in l. 906"),
 dict(species="CH3COCH3", name="acetone", group="ketone",
      pathway="chemistry", processes="Photolysis → CH3CO3 + CH3O2 (HOₓ source, mainly upper troposphere) and PAN; +OH → RO2 → CH3CO3, CH2O. Lifetime ~weeks.",
      direct_indirect="Indirect: upper-tropospheric HOₓ and PAN → O₃ (where O₃ is radiatively most effective).",
      linearity="Linear in HOₓ source; O₃ gain NOₓ-limited.", sign="🔥 small (upper-tropospheric O₃)",
      ref="chem_mech.in l. 607, 940"),
 dict(species="HCOOH", name="formic acid", group="organic acid",
      pathway="chemistry (minor)", processes="+OH → HO₂ + CO₂ (slow); mostly deposition.",
      direct_indirect="Indirect, negligible.", linearity="Linear.", sign="≈ 0", ref="chem_mech.in l. 875"),
 dict(species="HCN", name="hydrogen cyanide", group="nitrile",
      pathway="chemistry (minor)", processes="+OH (slow, termolecular) and +O(¹D) → HO₂/OH; lifetime months; wet deposition.",
      direct_indirect="Indirect, negligible.", linearity="Linear.", sign="≈ 0", ref="chem_mech.in l. 874, 882"),
 dict(species="CO", name="carbon monoxide (biogenic)", group="inorganic",
      pathway="chemistry", processes="+OH → CO₂ + HO₂ (lifetime ~2 months); HO₂ + NO → O₃; main background OH sink.",
      direct_indirect="Indirect: OH sink (→ CH₄ lifetime, largely suppressed by the CH₄ LBC, F8) and O₃.",
      linearity="Linear (long-lived, well-mixed); O₃ part NOₓ-limited.",
      sign="🔥 small (O₃; CH₄ part muted in our runs)", ref="chem_mech.in l. 873, 883"),
 dict(species="C2H6", name="ethane", group="alkane",
      pathway="chemistry", processes="+OH → C2H5O2 → CH3CHO → PAN (lifetime ~2 months); +Cl.",
      direct_indirect="Indirect: PAN/O₃.", linearity="Linear; O₃ part NOₓ-limited.", sign="≈ 0 (tiny flux)", ref="chem_mech.in l. 897–898"),
 dict(species="C3H8", name="propane", group="alkane",
      pathway="chemistry", processes="+OH → C3H7O2 → acetone, CH3CHO.",
      direct_indirect="Indirect: O₃/PAN.", linearity="Linear.", sign="≈ 0 (tiny flux)", ref="chem_mech.in l. 923–927"),
 dict(species="BIGALK", name="lumped C≥4 alkanes (butanes, pentanes, …)", group="alkane",
      pathway="chemistry", processes="+OH → ALKO2 → MEK, CH3CHO, acetone, alkyl nitrates (+ NO → O₃).",
      direct_indirect="Indirect: O₃/PAN.", linearity="Linear; O₃ part NOₓ-limited.", sign="≈ 0 (tiny flux)", ref="chem_mech.in l. 976–981"),
 dict(species="TOLUENE", name="toluene", group="aromatic",
      pathway="chemistry only (no SOA in this version)", processes="+OH → cresol, TOLO2 → glyoxal, methylglyoxal, BIGALDs (+ NO → O₃). VBS SOA yield commented out; glyoxal has no aerosol uptake.",
      direct_indirect="Indirect: O₃.", linearity="Linear; O₃ part NOₓ-limited.", sign="≈ 0 (tiny flux)", ref="chem_mech.in l. 1050–1052"),
]
info = pd.DataFrame(ROWS).set_index("species")
assert set(info.index) == set(totals.index), set(info.index) ^ set(totals.index)

In [ ]:
cols = {"global CTRL_PD": "global PD", "global CTRL_FUT": "global FUT",
        "modified CTRL_PD": "LCC cells CTRL_PD", "modified LCC_PD": "LCC cells LCC_PD",
        "modified CTRL_FUT": "LCC cells CTRL_FUT", "modified LCC_FUT": "LCC cells LCC_FUT",
        "modified FUT/PD": "LCC cells FUT/PD (CTRL)"}
table = (totals[list(cols)].rename(columns=cols).join(info)
         .sort_values("global PD", ascending=False)
         [["name", "group", *cols.values(), "pathway", "processes", "direct_indirect", "linearity", "sign", "ref"]])
table.to_csv("bvoc_species_processes.csv")
(table.style.format("{:.3g}", subset=list(cols.values()))
      .set_properties(**{"text-align": "left", "vertical-align": "top", "white-space": "normal"})
      .set_properties(subset=["processes", "direct_indirect", "linearity"], **{"min-width": "280px"})
      .set_table_styles([{"selector": "th", "props": "text-align:left; vertical-align:bottom"}]))

## 3. Compact view
Which species can act on the radiation budget through which channel, ranked by global PD emission.

In [ ]:
compact = pd.DataFrame({
    "name": table["name"],
    "Tg/yr PD → FUT (global)": [f"{a:.3g} → {b:.3g}" for a, b in zip(table["global PD"], table["global FUT"])],
    "SOA nucleation": ["✔ (SOA_LV)" if s == "MTERP" else "" for s in table.index],
    "SOA coating": ["✔" if s in ("MTERP", "ISOP") else "" for s in table.index],
    "oxidants (OH/O₃)": ["✔✔" if s in ("ISOP", "MTERP", "BCARY") else "✔" for s in table.index],
    "O₃ production / PAN": ["✔" if s not in ("CH3COOH", "HCOOH", "HCN") else "" for s in table.index],
    "sign of an increase": table["sign"],
})
display(compact)

## 4. Reading notes
- **Only ISOP and MTERP matter for aerosol in this model** (F1). MTERP is the only source of nucleating organics; ISOP only coats. Both cut OH, which lowers SO₂ → H₂SO₄ and thus the sulphate part of nucleation (indirect, opposite sign).
- **Non-linear by construction (a):** the Lehtinen survival factor is exp(−γ·d₁·CoagS/GR), so where GR is small (little organic vapour) a modest MTERP increase raises the 12 nm formation rate much faster than linearly; where GR is already large the factor → 1 and the gain becomes linear in J, then limited by the extra condensation sink that the new SOA itself creates. The ORG nucleation term only acts below max(PBLH, 500 m); the GR boost to survival applies at all levels (also to binary nucleation aloft).
- **Oxidant feedback (b):** the SOA tracer reactions use k·[VOC]·[oxidant]; when emissions rise strongly (FUT ≈ 2–4× PD), OH and O₃ in the boreal PBL are drawn down, so SOA per Tg of MTERP falls. Compare FUT/PD SOA burden with FUT/PD MTERP emission to see this.
- **Ozone (b):** boreal regions are mostly NOₓ-limited, so the many small oxygenated VOCs (CH3OH, acetone, acids) have small O₃ effects there; their effect is larger downwind of polluted areas and via PAN in the free troposphere.
- **CH₄ (a + b):** with CH₄ held by its lower boundary, the OH-suppression → CH₄ warming that dominates many published isoprene estimates is mostly absent in our runs, which shifts the BVOC net effect toward cooling compared with full-CH₄ studies.
- The signs are for each species alone. In the fBVOC contrasts all 19 change together, so cross-effects (e.g. BCARY taking O₃ from MTERP, ISOP taking OH from MTERP and SO₂) apply.